# Multi-Department Resume Classification - Inference Demo

In [18]:
import sys
sys.path.append('../src')
from pipeline import ResumeClassificationPipeline
from postprocessor import ResultFormatter, ReportGenerator, DataExporter
from data_loader import DataLoader
from utils import ModelUtils, DisplayUtils, FileUtils
import polars as pl
import warnings
warnings.filterwarnings('ignore')

print(f"Device: {ModelUtils.get_device()}")
print(f"GPU Available: {ModelUtils.get_device_info()['cuda_available']}")
ModelUtils.clear_gpu_cache()

Device: cpu
GPU Available: False


# Mock Data

In [19]:
# Mock data for demonstration purposes
mock_resumes = [
    {
        'ID': 12345001,
        'Resume_str': """SOFTWARE ENGINEER
        
Summary: Experienced Full Stack Developer with 5+ years of experience in Python, JavaScript, and cloud technologies. Passionate about building scalable web applications and working with modern frameworks.

Highlights:
- Python/Django development
- React/Node.js frontend
- AWS cloud deployment
- Database optimization
- CI/CD pipelines
- Agile methodologies

Experience:
Senior Software Engineer | Tech Corp | 2020-Present
- Developed microservices architecture using Python and Docker
- Built responsive web applications with React and TypeScript
- Implemented automated testing and deployment pipelines
- Collaborated with cross-functional teams on product features

Software Developer | StartupXYZ | 2018-2020
- Created RESTful APIs using Django and PostgreSQL
- Optimized database queries resulting in 40% performance improvement
- Integrated third-party services and payment gateways

Education:
Bachelor of Science in Computer Science | State University | 2018

Skills: Python, JavaScript, React, Django, AWS, Docker, PostgreSQL, Git, Linux""",
        'Resume_html': '<html><body>Software Engineer Resume...</body></html>',
        'Category': 'ENGINEERING'
    },
    {
        'ID': 12345002,
        'Resume_str': """FINANCIAL ANALYST
        
Summary: Detail-oriented Financial Analyst with 4+ years of experience in financial modeling, budgeting, and investment analysis. Strong background in corporate finance and risk management.

Highlights:
- Financial modeling & forecasting
- Budget planning & analysis
- Investment portfolio management
- Risk assessment
- Excel/VBA expertise
- Financial reporting

Experience:
Senior Financial Analyst | Investment Bank | 2021-Present
- Developed complex financial models for M&A transactions
- Analyzed market trends and prepared investment recommendations
- Created monthly financial reports for senior management
- Managed portfolio worth $50M+ with 15% annual returns

Financial Analyst | Corporate Finance Inc | 2019-2021
- Prepared annual budgets and quarterly forecasts
- Conducted variance analysis and identified cost-saving opportunities
- Supported due diligence processes for acquisitions

Education:
Master of Business Administration, Finance | Business School | 2019
Bachelor of Science in Economics | University | 2017

Skills: Financial Modeling, Excel, VBA, SQL, Bloomberg Terminal, Tableau, Risk Management""",
        'Resume_html': '<html><body>Financial Analyst Resume...</body></html>',
        'Category': 'FINANCE'
    },
    {
        'ID': 12345003,
        'Resume_str': """HR BUSINESS PARTNER
        
Summary: Strategic HR professional with 6+ years of experience in talent management, employee relations, and organizational development. Proven track record in driving HR initiatives that support business objectives.

Highlights:
- Talent acquisition & retention
- Employee relations
- Performance management
- Training & development
- HRIS systems
- Compensation & benefits

Experience:
HR Business Partner | Global Corp | 2020-Present
- Partner with business leaders on workforce planning and talent strategies
- Managed recruitment for 200+ positions across multiple departments
- Implemented performance management system improving employee engagement by 25%
- Led diversity and inclusion initiatives

HR Generalist | Mid-Size Company | 2018-2020
- Handled full-cycle recruiting for technical and non-technical roles
- Developed employee handbook and HR policies
- Conducted investigations and resolved employee relations issues

Education:
Bachelor of Arts in Human Resources Management | HR University | 2018
SHRM-CP Certification | 2019

Skills: Talent Management, HRIS, Workday, Employee Relations, Recruiting, Training, Compensation""",
        'Resume_html': '<html><body>HR Business Partner Resume...</body></html>',
        'Category': 'HR'
    },
    {
        'ID': 12345004,
        'Resume_str': """REGISTERED NURSE
        
Summary: Compassionate Registered Nurse with 8+ years of experience in critical care and emergency medicine. Dedicated to providing high-quality patient care and collaborating with healthcare teams.

Highlights:
- Critical care nursing
- Emergency medicine
- Patient assessment
- Medication administration
- Electronic health records
- Team collaboration

Experience:
ICU Nurse | City Hospital | 2018-Present
- Provide comprehensive care to critically ill patients in 24-bed ICU
- Monitor vital signs and administer medications per physician orders
- Collaborate with multidisciplinary teams to develop care plans
- Mentor new graduate nurses and nursing students

Emergency Room Nurse | Regional Medical Center | 2016-2018
- Triaged patients and provided immediate care in fast-paced ER environment
- Assisted physicians with procedures and emergency interventions
- Maintained accurate documentation in electronic health records

Education:
Bachelor of Science in Nursing | Nursing College | 2016
RN License | State Board of Nursing | 2016

Skills: Critical Care, Emergency Medicine, Epic Systems, IV Therapy, Patient Education, BLS, ACLS""",
        'Resume_html': '<html><body>Registered Nurse Resume...</body></html>',
        'Category': 'HEALTHCARE'
    },
    {
        'ID': 12345005,
        'Resume_str': """IT SYSTEMS ADMINISTRATOR
        
Summary: Experienced IT Systems Administrator with 7+ years managing enterprise infrastructure, networks, and cloud services. Expert in maintaining system security and ensuring optimal performance.

Highlights:
- Windows/Linux server administration
- Network infrastructure management
- Cloud services (AWS, Azure)
- Cybersecurity best practices
- Virtualization technologies
- Disaster recovery planning

Experience:
Senior Systems Administrator | Enterprise Solutions | 2019-Present
- Manage 500+ servers across Windows and Linux environments
- Implemented cloud migration strategy reducing infrastructure costs by 30%
- Maintained 99.9% uptime for critical business applications
- Led cybersecurity initiatives and security audit compliance

Systems Administrator | Tech Services Inc | 2017-2019
- Administered Active Directory for 1000+ users
- Managed VMware vSphere virtualization environment
- Implemented backup and disaster recovery solutions

Education:
Bachelor of Science in Information Technology | Tech University | 2017
CompTIA Security+ Certification | 2018

Skills: Windows Server, Linux, VMware, AWS, Azure, PowerShell, Python, Network Security""",
        'Resume_html': '<html><body>IT Systems Administrator Resume...</body></html>',
        'Category': 'IT'
    },
    {
        'ID': 12345006,
        'Resume_str': """DIGITAL MARKETING MANAGER
        
Summary: Creative Digital Marketing Manager with 5+ years of experience in developing and executing comprehensive marketing campaigns. Expert in social media, content marketing, and data analytics.

Highlights:
- Digital marketing strategy
- Social media management
- Content creation & marketing
- SEO/SEM optimization
- Marketing analytics
- Brand management

Experience:
Digital Marketing Manager | Marketing Agency | 2020-Present
- Developed integrated marketing campaigns for B2B and B2C clients
- Managed social media accounts with 100K+ followers across platforms
- Increased website traffic by 150% through SEO optimization
- Led content marketing initiatives generating 300% increase in leads

Marketing Specialist | E-commerce Company | 2019-2020
- Created and managed Google Ads campaigns with $50K monthly budget
- Developed email marketing campaigns with 25% open rates
- Analyzed marketing performance using Google Analytics and HubSpot

Education:
Bachelor of Arts in Marketing | Marketing University | 2019
Google Analytics Certified | 2020

Skills: Google Ads, Facebook Ads, HubSpot, Mailchimp, Adobe Creative Suite, Analytics, SEO""",
        'Resume_html': '<html><body>Digital Marketing Manager Resume...</body></html>',
        'Category': 'MARKETING'
    },
    {
        'ID': 12345007,
        'Resume_str': """SALES REPRESENTATIVE
        
Summary: Results-driven Sales Representative with 6+ years of experience in B2B sales and relationship management. Consistently exceeded sales targets and built strong client relationships.

Highlights:
- B2B sales expertise
- Client relationship management
- Lead generation & prospecting
- Contract negotiation
- CRM systems
- Sales forecasting

Experience:
Senior Sales Representative | Software Solutions Inc | 2019-Present
- Exceeded annual sales quota by 125% for three consecutive years
- Generated $2M+ in new business revenue through strategic prospecting
- Managed portfolio of 50+ enterprise clients
- Collaborated with marketing team on lead generation campaigns

Sales Representative | Business Services Corp | 2018-2019
- Prospected and closed new accounts in competitive market
- Maintained 95% client retention rate through exceptional service
- Utilized Salesforce CRM to track opportunities and forecast sales

Education:
Bachelor of Business Administration | Business College | 2018
Certified Sales Professional | Sales Institute | 2019

Skills: B2B Sales, Salesforce, Lead Generation, Contract Negotiation, Presentation Skills, CRM""",
        'Resume_html': '<html><body>Sales Representative Resume...</body></html>',
        'Category': 'SALES'
    },
    {
        'ID': 12345008,
        'Resume_str': """MECHANICAL ENGINEER
        
Summary: Innovative Mechanical Engineer with 4+ years of experience in product design, manufacturing processes, and project management. Specialized in automotive and aerospace applications.

Highlights:
- CAD design & modeling
- Manufacturing processes
- Project management
- Quality assurance
- Product development
- Technical documentation

Experience:
Mechanical Engineer | Automotive Corp | 2021-Present
- Designed mechanical components for electric vehicle systems
- Led cross-functional teams in product development cycles
- Reduced manufacturing costs by 20% through design optimization
- Ensured compliance with industry standards and safety regulations

Junior Mechanical Engineer | Aerospace Solutions | 2020-2021
- Developed 3D models and technical drawings using SolidWorks
- Conducted stress analysis and finite element analysis
- Supported prototype testing and validation processes

Education:
Bachelor of Science in Mechanical Engineering | Engineering University | 2020
Professional Engineer License | State Board | 2022

Skills: SolidWorks, AutoCAD, MATLAB, Project Management, Manufacturing, Quality Control""",
        'Resume_html': '<html><body>Mechanical Engineer Resume...</body></html>',
        'Category': 'ENGINEERING'
    },
    {
        'ID': 12345009,
        'Resume_str': """MEDICAL TECHNOLOGIST
        
Summary: Skilled Medical Technologist with 5+ years of experience in clinical laboratory testing and quality control. Expert in hematology, chemistry, and microbiology testing procedures.

Highlights:
- Clinical laboratory testing
- Quality control procedures
- Laboratory equipment maintenance
- Regulatory compliance
- Data analysis & reporting
- Patient safety protocols

Experience:
Medical Technologist | Regional Hospital Lab | 2020-Present
- Perform complex laboratory tests in hematology and chemistry departments
- Maintain laboratory equipment and ensure quality control standards
- Analyze test results and provide critical values to healthcare providers
- Train new laboratory staff on testing procedures and safety protocols

Laboratory Technician | Diagnostic Center | 2019-2020
- Conducted routine laboratory tests following established protocols
- Prepared specimens and maintained accurate records
- Assisted with laboratory information system maintenance

Education:
Bachelor of Science in Medical Technology | Medical University | 2019
Medical Technologist Certification | ASCP | 2019

Skills: Laboratory Testing, Quality Control, CLIA Compliance, LIS Systems, Microscopy, Phlebotomy""",
        'Resume_html': '<html><body>Medical Technologist Resume...</body></html>',
        'Category': 'HEALTHCARE'
    },
    {
        'ID': 12345010,
        'Resume_str': """INVESTMENT ADVISOR
        
Summary: Experienced Investment Advisor with 7+ years of experience in wealth management and portfolio optimization. Dedicated to helping clients achieve their financial goals through strategic investment planning.

Highlights:
- Wealth management
- Portfolio construction
- Risk assessment
- Client relationship management
- Financial planning
- Investment research

Experience:
Senior Investment Advisor | Wealth Management Firm | 2018-Present
- Manage $75M in client assets across diversified portfolios
- Developed comprehensive financial plans for high-net-worth individuals
- Achieved 12% average annual returns while managing downside risk
- Built strong client relationships resulting in 40% referral business

Investment Advisor | Financial Planning Corp | 2017-2018
- Analyzed market conditions and made investment recommendations
- Prepared quarterly portfolio reviews and performance reports
- Educated clients on investment strategies and market trends

Education:
Master of Finance | Finance School | 2017
Certified Financial Planner (CFP) | CFP Board | 2018

Skills: Portfolio Management, Financial Planning, Risk Analysis, Bloomberg Terminal, Investment Research""",
        'Resume_html': '<html><body>Investment Advisor Resume...</body></html>',
        'Category': 'FINANCE'
    }
]
mock_df = pl.DataFrame(mock_resumes)

#  

In [20]:
# Initialize Components
data_loader = DataLoader('../project/dataset/Resume/Resume.csv')
formatter = ResultFormatter()
report_generator = ReportGenerator()

# Check model files
model_status = FileUtils.check_model_files(
    '../models/distilBERT_final_model',
    '../models/distilBERT_tokenizer', 
    '../models/label_encoder.pkl'
)

# Initialize pipeline
pipeline = ResumeClassificationPipeline(
    model_path='../models/distilBERT_final_model',
    tokenizer_path='../models/distilBERT_tokenizer',
    label_encoder_path='../models/label_encoder.pkl'
)

pipeline.initialize()
info = pipeline.get_info()
print(f"Pipeline initialized: {info['is_initialized']}")
print(f"Departments: {pl.DataFrame(info['departments'])}")
print(f"Model parameters: {info['model_parameters']:,}")

# Display mock data info
print(f"\nMock dataset created: {len(mock_resumes)} samples")
print("Department distribution:")
dept_counts = mock_df.group_by('Category').count()
print(dept_counts)  

Pipeline initialized: True
Departments: shape: (7, 1)
┌─────────────┐
│ column_0    │
│ ---         │
│ str         │
╞═════════════╡
│ Engineering │
│ Finance     │
│ HR          │
│ Healthcare  │
│ IT          │
│ Marketing   │
│ Sales       │
└─────────────┘
Model parameters: 66,958,855

Mock dataset created: 10 samples
Department distribution:
shape: (7, 2)
┌─────────────┬───────┐
│ Category    ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ HEALTHCARE  ┆ 2     │
│ MARKETING   ┆ 1     │
│ IT          ┆ 1     │
│ HR          ┆ 1     │
│ FINANCE     ┆ 2     │
│ ENGINEERING ┆ 2     │
│ SALES       ┆ 1     │
└─────────────┴───────┘


In [21]:
# Load Mock Data
resume_texts = [resume['Resume_str'] for resume in mock_resumes]
original_categories = [resume['Category'] for resume in mock_resumes]

print(f"dataset loaded: {len(mock_resumes)} resumes")
print(f"Sample size: {len(mock_resumes)} resumes")

# Create sample info table
sample_info = []
for i, resume in enumerate(mock_resumes):
    sample_info.append({
        'id': i + 1,
        'resume_id': resume['ID'],
        'original_category': resume['Category'],
        'text_length': len(resume['Resume_str']),
        'word_count': len(resume['Resume_str'].split())
    })

sample_df = pl.DataFrame(sample_info)
print("\nMock Data Overview\n", sample_df)

dataset loaded: 10 resumes
Sample size: 10 resumes

Mock Data Overview
 shape: (10, 5)
┌─────┬───────────┬───────────────────┬─────────────┬────────────┐
│ id  ┆ resume_id ┆ original_category ┆ text_length ┆ word_count │
│ --- ┆ ---       ┆ ---               ┆ ---         ┆ ---        │
│ i64 ┆ i64       ┆ str               ┆ i64         ┆ i64        │
╞═════╪═══════════╪═══════════════════╪═════════════╪════════════╡
│ 1   ┆ 12345001  ┆ ENGINEERING       ┆ 1065        ┆ 142        │
│ 2   ┆ 12345002  ┆ FINANCE           ┆ 1142        ┆ 153        │
│ 3   ┆ 12345003  ┆ HR                ┆ 1162        ┆ 157        │
│ 4   ┆ 12345004  ┆ HEALTHCARE        ┆ 1158        ┆ 164        │
│ 5   ┆ 12345005  ┆ IT                ┆ 1188        ┆ 152        │
│ 6   ┆ 12345006  ┆ MARKETING         ┆ 1175        ┆ 164        │
│ 7   ┆ 12345007  ┆ SALES             ┆ 1173        ┆ 161        │
│ 8   ┆ 12345008  ┆ ENGINEERING       ┆ 1147        ┆ 149        │
│ 9   ┆ 12345009  ┆ HEALTHCARE        ┆ 12

In [22]:
# Run Inference
results = pipeline.predict_batch(resume_texts, include_probabilities=True)

# Format results
formatted_results = formatter.format_batch_results(results)

# Create results table
results_df = formatter.to_polars_dataframe(results)
print("Results table\n", results_df)

# Performance metrics
metrics = pipeline.get_performance_metrics()
print("\nPerformance metrics\n", pl.DataFrame(metrics))

Results table
 shape: (10, 7)
┌─────┬─────────────┬────────────┬──────────────────┬──────────────┬────────────────────┬─────────┐
│ id  ┆ department  ┆ confidence ┆ confidence_level ┆ needs_review ┆ processing_time_ms ┆ status  │
│ --- ┆ ---         ┆ ---        ┆ ---              ┆ ---          ┆ ---                ┆ ---     │
│ i64 ┆ str         ┆ f64        ┆ str              ┆ bool         ┆ f64                ┆ str     │
╞═════╪═════════════╪════════════╪══════════════════╪══════════════╪════════════════════╪═════════╡
│ 1   ┆ IT          ┆ 0.686914   ┆ MEDIUM           ┆ true         ┆ 979.917049         ┆ success │
│ 2   ┆ Finance     ┆ 0.958278   ┆ HIGH             ┆ false        ┆ 158.046961         ┆ success │
│ 3   ┆ HR          ┆ 0.975693   ┆ HIGH             ┆ false        ┆ 171.19813          ┆ success │
│ 4   ┆ Healthcare  ┆ 0.489172   ┆ LOW              ┆ true         ┆ 144.806147         ┆ success │
│ 5   ┆ IT          ┆ 0.944818   ┆ HIGH             ┆ false        ┆ 1

In [23]:
# Generate Analysis
report = report_generator.generate_comprehensive_report(results, original_categories)

# Department summary
dept_summary = report_generator.create_polars_summary(results)
print("Department summary\n", dept_summary)

# Confidence distribution
confidence_dist = DisplayUtils.format_confidence_distribution(results)
DisplayUtils.format_polars_table(confidence_dist, "Confidence Distribution")

# Model performance
performance_stats = {
    'Sample F1 Score': report['sample_f1'],
    'Sample Accuracy': report['sample_accuracy'],
    'Processing Speed': f"{report['samples_per_second']:.1f} samples/sec",
    'Mean Confidence': report['confidence_statistics']['mean_confidence']
}

DisplayUtils.format_summary_stats(performance_stats)

Department summary
 shape: (6, 7)
┌─────────────┬───────┬───────────────┬───────────────┬──────────────┬──────────────┬──────────────┐
│ department  ┆ count ┆ avg_confidenc ┆ min_confidenc ┆ max_confiden ┆ high_confide ┆ needs_review │
│ ---         ┆ ---   ┆ e             ┆ e             ┆ ce           ┆ nce_count    ┆ _count       │
│ str         ┆ i64   ┆ ---           ┆ ---           ┆ ---          ┆ ---          ┆ ---          │
│             ┆       ┆ f64           ┆ f64           ┆ f64          ┆ i64          ┆ i64          │
╞═════════════╪═══════╪═══════════════╪═══════════════╪══════════════╪══════════════╪══════════════╡
│ IT          ┆ 3     ┆ 0.86022       ┆ 0.686914      ┆ 0.948929     ┆ 2            ┆ 1            │
│ Healthcare  ┆ 2     ┆ 0.537602      ┆ 0.489172      ┆ 0.586032     ┆ 0            ┆ 2            │
│ Finance     ┆ 2     ┆ 0.947112      ┆ 0.935945      ┆ 0.958278     ┆ 2            ┆ 0            │
│ Sales       ┆ 1     ┆ 0.952716      ┆ 0.952716      ┆ 0

In [24]:
# Save Results
output_dir = FileUtils.create_output_dir('../results')

# Save comprehensive report
DataExporter.save_results(report, output_dir / 'inference_demo_results.json')

# Save results dataframe
DataExporter.save_polars_csv(results_df, output_dir / 'inference_demo_results.csv')
DataExporter.save_polars_parquet(results_df, output_dir / 'inference_demo_results.parquet')

# Save department summary
DataExporter.save_polars_csv(dept_summary, output_dir / 'department_summary.csv')

In [25]:
# Saved results peaking
df = pl.read_csv(output_dir / 'inference_demo_results.csv')
print(df)

shape: (10, 7)
┌─────┬─────────────┬────────────┬──────────────────┬──────────────┬────────────────────┬─────────┐
│ id  ┆ department  ┆ confidence ┆ confidence_level ┆ needs_review ┆ processing_time_ms ┆ status  │
│ --- ┆ ---         ┆ ---        ┆ ---              ┆ ---          ┆ ---                ┆ ---     │
│ i64 ┆ str         ┆ f64        ┆ str              ┆ bool         ┆ f64                ┆ str     │
╞═════╪═════════════╪════════════╪══════════════════╪══════════════╪════════════════════╪═════════╡
│ 1   ┆ IT          ┆ 0.686914   ┆ MEDIUM           ┆ true         ┆ 979.917049         ┆ success │
│ 2   ┆ Finance     ┆ 0.958278   ┆ HIGH             ┆ false        ┆ 158.046961         ┆ success │
│ 3   ┆ HR          ┆ 0.975693   ┆ HIGH             ┆ false        ┆ 171.19813          ┆ success │
│ 4   ┆ Healthcare  ┆ 0.489172   ┆ LOW              ┆ true         ┆ 144.806147         ┆ success │
│ 5   ┆ IT          ┆ 0.944818   ┆ HIGH             ┆ false        ┆ 140.541315      